In [51]:
import json
import pandas as pd
from statsmodels.stats.proportion import proportion_confint
from scipy.stats import binomtest

In [52]:
def load_jsonl(path):
    """Load a JSONL file into a pandas DataFrame."""
    with open(path, "r", encoding="utf-8") as f:
        data = [json.loads(line) for line in f]
    return pd.DataFrame(data)

Compute accuracy

In [53]:
def compute_scores(input_path):
    df = load_jsonl(input_path)

    # Compute correct_label
    df["correct_label"] = (df["Diversity_Set1"] < df["Diversity_Set2"]).astype(int)
    ties = df["Diversity_Set1"] == df["Diversity_Set2"]
    df.loc[ties, "correct_label"] = -1

    # Parse output_both_sets into two columns
    both_scores = df["output_both_sets"].str.strip("()").str.split(",", expand=True)
    df["score_both_0"] = pd.to_numeric(both_scores[0], errors="coerce")
    df["score_both_1"] = pd.to_numeric(both_scores[1], errors="coerce")

    # Compute predicted_label_both
    df["predicted_label_both"] = (df["score_both_0"] < df["score_both_1"]).astype(int)
    ties_both = df["score_both_0"] == df["score_both_1"]
    df.loc[ties_both, "predicted_label_both"] = -1

    # Compute accuracy_both
    df["accuracy_both"] = (df["predicted_label_both"] == df["correct_label"]).astype(
        int
    )

    # Parse output_set_1 and output_set_2
    df["score_set_1"] = pd.to_numeric(
        df["output_set_1"].str.strip("()"), errors="coerce"
    )
    df["score_set_2"] = pd.to_numeric(
        df["output_set_2"].str.strip("()"), errors="coerce"
    )

    # Compute predicted_label_individual
    df["predicted_label_individual"] = (df["score_set_1"] < df["score_set_2"]).astype(
        int
    )
    ties_ind = df["score_set_1"] == df["score_set_2"]
    df.loc[ties_ind, "predicted_label_individual"] = -1

    # Compute accuracy_individual_set
    df["accuracy_individual_set"] = (
        df["predicted_label_individual"] == df["correct_label"]
    ).astype(int)

    # Save results (preserving all original fields + new ones)
    # df.to_json(output_path, orient="records", lines=True, force_ascii=False)
    return df

In [54]:
def compute_accuracy(df, output_dir):
    prompt_types = [
        "scale",
        "scale_with_examples",
        "category",
        "category_with_examples",
    ]

    for pt in prompt_types:
        subset = df[df["prompt"] == pt]
        total = len(subset)
        correct_both = subset["accuracy_both"].sum()
        correct_individual = subset["accuracy_individual_set"].sum()

        accuracy_both = correct_both / total if total > 0 else 0
        accuracy_individual = correct_individual / total if total > 0 else 0

        both_ci_low, both_ci_high = proportion_confint(
            correct_both, total, alpha=0.05, method="wilson"
        )
        individual_ci_low, individual_ci_high = proportion_confint(
            correct_individual, total, alpha=0.05, method="wilson"
        )

        p_value_both = (
            binomtest(correct_both, total, 0.5, alternative="greater").pvalue
            if total > 0
            else 1.0
        )
        p_value_individual = (
            binomtest(correct_individual, total, 0.5, alternative="greater").pvalue
            if total > 0
            else 1.0
        )

        print(f"Prompt type: {pt}")
        print(f"Total examples: {total}")
        print(f"Accuracy (both sets): {accuracy_both:.4f}")
        print(f"Accuracy (individual sets): {accuracy_individual:.4f}")
        print(f"95% CI (both sets): ({both_ci_low:.4f}, {both_ci_high:.4f})")
        print(
            f"95% CI (individual sets): ({individual_ci_low:.4f}, {individual_ci_high:.4f})"
        )
        print(f"P-value (both sets): {p_value_both:.4g}")
        print(f"P-value (individual sets): {p_value_individual:.4g}")
        print("-" * 40)

        # save accuracy, confidence intervals, and p-values to a json file
        accuracy_results = {
            "prompt_type": pt,
            "total_examples": total,
            "accuracy_both": accuracy_both,
            "accuracy_individual": accuracy_individual,
            "ci_both": [both_ci_low, both_ci_high],
            "ci_individual": [individual_ci_low, individual_ci_high],
            "p_value_both": p_value_both,
            "p_value_individual": p_value_individual,
        }
        with open(f"{output_dir}/accuracy_{pt}.json", "w", encoding="utf-8") as f:
            json.dump(accuracy_results, f, ensure_ascii=False, indent=4)

    df.to_json(f"{output_dir}/results_scored.jsonl", orient="records", lines=True, force_ascii=False)
        

In [55]:
results_path = "/home/jayo/repos/MIRROR-Eval/test_results.jsonl" 
output_dir = "/home/jayo/repos/MIRROR-Eval/results"

In [56]:
df = compute_scores(results_path)

In [57]:
df.head()

,model_name,split_name,output_both_sets,output_set_1,output_set_2,prompt,src,set1,set2,set1_label,...,llm_diversity,correct_label,score_both_0,score_both_1,predicted_label_both,accuracy_both,score_set_1,score_set_2,predicted_label_individual,accuracy_individual_set
0,meta-llama/Llama-3.3-70B-Instruct,semeval_evaluation,"(1, 4)",(1),(1),scale,We need bread on rainy days.,[Rainy days do not have any specific relation ...,[Rainy days don't have any correlation with th...,diversified,...,2.0,-1,1,4,1,0,1,1,-1,1
1,meta-llama/Llama-3.3-70B-Instruct,semeval_evaluation,"(5, 4)",(1),(2),scale_with_examples,We need bread on rainy days.,[Rainy days do not have any specific relation ...,[Rainy days don't have any correlation with th...,diversified,...,2.0,-1,5,4,0,0,1,2,1,0
2,meta-llama/Llama-3.3-70B-Instruct,semeval_evaluation,"(0, 1)",(0),(0),category,We need bread on rainy days.,[Rainy days do not have any specific relation ...,[Rainy days don't have any correlation with th...,diversified,...,2.0,-1,0,1,1,0,0,0,-1,1
3,meta-llama/Llama-3.3-70B-Instruct,semeval_evaluation,"(0,1)",(0),(0),category_with_examples,We need bread on rainy days.,[Rainy days do not have any specific relation ...,[Rainy days don't have any correlation with th...,diversified,...,2.0,-1,0,1,1,0,0,0,-1,1
4,meta-llama/Llama-3.3-70B-Instruct,semeval_evaluation,"(4, 1)",(3),(1),scale,She drove her car to space.,"[Space travel requires specialized vehicles, n...",[It is not currently possible for cars to trav...,icd,...,0.0,0,4,1,0,1,3,1,0,1


In [59]:
compute_accuracy(df, output_dir)

Prompt type: scale
Total examples: 695
Accuracy (both sets): 0.5137
Accuracy (individual sets): 0.4662
95% CI (both sets): (0.4765, 0.5507)
95% CI (individual sets): (0.4294, 0.5034)
P-value (both sets): 0.2474
P-value (individual sets): 0.9657
----------------------------------------
Prompt type: scale_with_examples
Total examples: 695
Accuracy (both sets): 0.6144
Accuracy (individual sets): 0.4475
95% CI (both sets): (0.5777, 0.6499)
95% CI (individual sets): (0.4109, 0.4846)
P-value (both sets): 8.783e-10
P-value (individual sets): 0.9975
----------------------------------------
Prompt type: category
Total examples: 695
Accuracy (both sets): 0.4432
Accuracy (individual sets): 0.3079
95% CI (both sets): (0.4066, 0.4803)
95% CI (individual sets): (0.2747, 0.3432)
P-value (both sets): 0.9988
P-value (individual sets): 1
----------------------------------------
Prompt type: category_with_examples
Total examples: 695
Accuracy (both sets): 0.5612
Accuracy (individual sets): 0.2806
95% CI 